In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score

sys.path.insert(0, os.path.abspath('..'))

from src.models.predict import (
    load_cat_models,
    load_lgb_models,
    load_meta_model,
    load_xgb_models,
    make_submission_frame,
    predict_with_meta_model,
    predict_with_tta,
    wrap_lgb_model,
)
from src.models.train import (
    LABEL_MAP,
    REVERSE_MAP,
    build_pseudo_labeled_training_data,
    cross_validate_models,
    encode_labels,
    save_meta_model,
    save_models,
    train_stacking_meta_model,
)

warnings.filterwarnings('ignore')

project_root_directory = Path('..').resolve()
processed_data_directory = project_root_directory / 'data' / 'processed'
trained_models_directory = project_root_directory / 'models'
pseudo_labeled_models_directory = (
    project_root_directory / 'models' / 'pseudo_labeled_v2'
)
outputs_directory = project_root_directory / 'outputs'

outputs_directory.mkdir(parents=True, exist_ok=True)

training_features = pd.read_csv(processed_data_directory / 'X_train.csv')
training_labels = pd.read_csv(processed_data_directory / 'y_train.csv').squeeze()
test_features = pd.read_csv(processed_data_directory / 'X_test.csv')
raw_test_data = pd.read_csv(
    project_root_directory / 'data' / 'raw' / 'test.csv'
)

print(f'X_train shape: {training_features.shape}')
print(f'y_train shape: {training_labels.shape}')
print(f'X_test shape: {test_features.shape}')
print(f'Test IDs: {len(raw_test_data)} records')
print('\nClass distribution (training):')
print(training_labels.value_counts().to_string())

In [ ]:
import importlib
import src.models.predict as predict_module
import src.models.train as train_module

importlib.reload(predict_module)
importlib.reload(train_module)

print('Reloaded src.models.predict and src.models.train successfully')

In [ ]:
print('Loading base models trained in Notebook 03...')

lightgbm_models = load_lgb_models(str(trained_models_directory))
xgboost_models = load_xgb_models(str(trained_models_directory))
catboost_models = load_cat_models(str(trained_models_directory))

print(f'Loaded {len(lightgbm_models)} LightGBM models')
print(f'Loaded {len(xgboost_models)} XGBoost models')
print(f'Loaded {len(catboost_models)} CatBoost models')
print(
    f'\nTotal base models: '
    f'{len(lightgbm_models) + len(xgboost_models) + len(catboost_models)}'
)

In [ ]:
print('\n' + '=' * 60)
print('PSEUDO-LABEL GENERATION STRATEGY')
print('=' * 60)

lightgbm_models = [
    wrap_lgb_model(model)
    if hasattr(model, 'predict') and not hasattr(model, 'predict_proba')
    else model
    for model in lightgbm_models
]

test_features = test_features.copy()

test_predictions_lightgbm = np.zeros((len(test_features), 3))
test_predictions_xgboost = np.zeros((len(test_features), 3))
test_predictions_catboost = np.zeros((len(test_features), 3))

for lightgbm_model in lightgbm_models:
    test_predictions_lightgbm += lightgbm_model.predict_proba(test_features)
test_predictions_lightgbm /= len(lightgbm_models)

for xgboost_model in xgboost_models:
    test_predictions_xgboost += xgboost_model.predict_proba(test_features)
test_predictions_xgboost /= len(xgboost_models)

for catboost_model in catboost_models:
    test_predictions_catboost += catboost_model.predict_proba(test_features)
test_predictions_catboost /= len(catboost_models)

ensemble_test_probabilities = (
    test_predictions_lightgbm * 0.60
    + test_predictions_xgboost * 0.35
    + test_predictions_catboost * 0.05
)

print('Test set predictions generated via weighted ensemble')
print(f'  Final ensemble shape: {ensemble_test_probabilities.shape}')
print(
    f'  Probability range: '
    f'[{ensemble_test_probabilities.min():.4f}, {ensemble_test_probabilities.max():.4f}]'
)

In [ ]:
print('\n' + '=' * 60)
print('BUILDING PSEUDO-LABELED TRAINING DATA')
print('=' * 60)

class_confidence_thresholds = {0: 0.995, 1: 0.995, 2: 0.95}

(
    augmented_training_features,
    augmented_training_labels,
    selected_pseudo_label_mask,
    selected_pseudo_labels,
) = build_pseudo_labeled_training_data(
    training_features,
    training_labels,
    test_features,
    ensemble_test_probabilities,
    class_thresholds=class_confidence_thresholds,
)

print(f'\nOriginal training set size: {len(training_features):,}')
print(
    f'High-confidence pseudo-labels selected: '
    f'{selected_pseudo_label_mask.sum():,}'
)
print(f'Augmented training set size: {len(augmented_training_features):,}')
print('\nOriginal class distribution:')
print(training_labels.value_counts().to_string())
print('\nAugmented class distribution:')
print(augmented_training_labels.value_counts().to_string())

In [ ]:
print('\n' + '=' * 60)
print('RETRAINING BASE MODELS ON PSEUDO-LABELED DATA (10-FOLD CV)')
print('=' * 60)

lightgbm_hyperparameters = {
    'learning_rate': 0.038,
    'num_leaves': 224,
    'max_depth': 12,
    'min_child_samples': 21,
    'feature_fraction': 0.772,
    'bagging_fraction': 0.945,
    'bagging_freq': 8,
    'reg_alpha': 0.321,
    'reg_lambda': 0.424,
}

xgboost_hyperparameters = {
    'learning_rate': 0.068,
    'max_depth': 8,
    'subsample': 0.921,
    'colsample_bytree': 0.805,
    'reg_alpha': 0.034,
    'reg_lambda': 0.365,
    'min_child_weight': 6,
}

(
    pseudo_labeled_lightgbm_models,
    pseudo_labeled_xgboost_models,
    pseudo_labeled_catboost_models,
    _,
    out_of_fold_predictions_pseudo_labeled_lightgbm,
    out_of_fold_predictions_pseudo_labeled_xgboost,
    out_of_fold_predictions_pseudo_labeled_catboost,
    _,
) = cross_validate_models(
    augmented_training_features,
    augmented_training_labels,
    n_splits=10,
    lgb_params=lightgbm_hyperparameters,
    xgb_params=xgboost_hyperparameters,
    include_mlp=False,
)

print('\nRetraining completed with 10-fold cross-validation')
print(f'Trained {len(pseudo_labeled_lightgbm_models)} LightGBM models')
print(f'Trained {len(pseudo_labeled_xgboost_models)} XGBoost models')
print(f'Trained {len(pseudo_labeled_catboost_models)} CatBoost models')

In [ ]:
import re
from pathlib import Path

log_directory = project_root_directory / 'logs'
log_files = sorted(log_directory.glob('training_*.log'), key=lambda p: p.stat().st_mtime)
assert log_files, f'No training logs found in {log_directory}'
latest_log_path = log_files[-1]
log_text = latest_log_path.read_text(encoding='utf-8', errors='ignore')

fold_pattern = re.compile(
    r'LightGBM BA: ([\d.]+).*?XGBoost BA: ([\d.]+).*?CatBoost BA: ([\d.]+).*?'
    r'Fold \d+ Ensemble BA: ([\d.]+)',
    re.DOTALL,
)
per_fold_scores = [
    tuple(map(float, match.groups())) for match in fold_pattern.finditer(log_text)
]
print(f'Parsed {len(per_fold_scores)} folds from {latest_log_path.name}')

if len(per_fold_scores) == 10:
    fold_numbers = list(range(1, 11))
    lightgbm_fold_scores, xgboost_fold_scores, catboost_fold_scores, ensemble_fold_scores = zip(
        *per_fold_scores
    )

    plt.figure(figsize=(8, 5))
    plt.plot(fold_numbers, lightgbm_fold_scores, marker='o', label='LightGBM', color='#1f77b4')
    plt.plot(fold_numbers, xgboost_fold_scores, marker='o', label='XGBoost', color='#ff7f0e')
    plt.plot(fold_numbers, catboost_fold_scores, marker='o', label='CatBoost', color='#2ca02c')
    plt.plot(fold_numbers, ensemble_fold_scores, marker='o', label='Ensemble', color='#d62728')
    plt.xlabel('Fold')
    plt.ylabel('Balanced Accuracy')
    plt.title('Balanced Accuracy per Fold on Augmented Set (Notebook 04)')
    plt.xticks(fold_numbers)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        outputs_directory / 'fold_balanced_accuracy_augmented.png',
        dpi=200,
        bbox_inches='tight',
    )
    plt.show()
    print(f'Figure saved to {outputs_directory / "fold_balanced_accuracy_augmented.png"}')
else:
    print(
        f'⚠ Expected 10 folds but parsed {len(per_fold_scores)} — check that '
        f'{latest_log_path.name} is the log from THIS retraining run before plotting.'
    )


In [ ]:
original_training_count = len(training_features)
encoded_original_training_targets = encode_labels(training_labels)

original_out_of_fold_predictions_lightgbm = (
    out_of_fold_predictions_pseudo_labeled_lightgbm[:original_training_count]
)
original_out_of_fold_predictions_xgboost = (
    out_of_fold_predictions_pseudo_labeled_xgboost[:original_training_count]
)
original_out_of_fold_predictions_catboost = (
    out_of_fold_predictions_pseudo_labeled_catboost[:original_training_count]
)

balanced_accuracy_lightgbm = balanced_accuracy_score(
    encoded_original_training_targets,
    original_out_of_fold_predictions_lightgbm.argmax(axis=1),
)
balanced_accuracy_xgboost = balanced_accuracy_score(
    encoded_original_training_targets,
    original_out_of_fold_predictions_xgboost.argmax(axis=1),
)
balanced_accuracy_catboost = balanced_accuracy_score(
    encoded_original_training_targets,
    original_out_of_fold_predictions_catboost.argmax(axis=1),
)

blended_original_out_of_fold_probabilities = (
    original_out_of_fold_predictions_lightgbm * 0.60
    + original_out_of_fold_predictions_xgboost * 0.35
    + original_out_of_fold_predictions_catboost * 0.05
)
balanced_accuracy_blended_ensemble = balanced_accuracy_score(
    encoded_original_training_targets,
    blended_original_out_of_fold_probabilities.argmax(axis=1),
)

added_pseudo_label_count = (
    len(augmented_training_features) - original_training_count
)
pseudo_label_percentage = (
    100
    * added_pseudo_label_count
    / len(augmented_training_features)
)

print(
    f'Pseudo-labeled rows added: {added_pseudo_label_count:,} '
    f'(out of {len(augmented_training_features):,} total combined rows, '
    f'{pseudo_label_percentage:.1f}%)'
)
print()
print('HONEST OOF BA (original labeled rows only):')
print(f'  LightGBM : {balanced_accuracy_lightgbm:.4f}')
print(f'  XGBoost  : {balanced_accuracy_xgboost:.4f}')
print(f'  CatBoost : {balanced_accuracy_catboost:.4f}')
print(f'  Blend (0.60/0.35/0.05): {balanced_accuracy_blended_ensemble:.4f}')
print()
print(
    '(vs. the pseudo-label-inflated full-set OOF BA reported in the next cell: ~0.9727)'
)

In [ ]:
encoded_augmented_targets = encode_labels(augmented_training_labels)

inflated_balanced_accuracy_lightgbm = balanced_accuracy_score(
    encoded_augmented_targets,
    out_of_fold_predictions_pseudo_labeled_lightgbm.argmax(axis=1),
)
inflated_balanced_accuracy_xgboost = balanced_accuracy_score(
    encoded_augmented_targets,
    out_of_fold_predictions_pseudo_labeled_xgboost.argmax(axis=1),
)
inflated_balanced_accuracy_catboost = balanced_accuracy_score(
    encoded_augmented_targets,
    out_of_fold_predictions_pseudo_labeled_catboost.argmax(axis=1),
)
inflated_blended_out_of_fold_probabilities = (
    out_of_fold_predictions_pseudo_labeled_lightgbm * 0.60
    + out_of_fold_predictions_pseudo_labeled_xgboost * 0.35
    + out_of_fold_predictions_pseudo_labeled_catboost * 0.05
)
inflated_balanced_accuracy_ensemble = balanced_accuracy_score(
    encoded_augmented_targets,
    inflated_blended_out_of_fold_probabilities.argmax(axis=1),
)

model_names = ['LightGBM', 'XGBoost', 'CatBoost', 'Blend/Ensemble']
inflated_scores = [
    inflated_balanced_accuracy_lightgbm,
    inflated_balanced_accuracy_xgboost,
    inflated_balanced_accuracy_catboost,
    inflated_balanced_accuracy_ensemble,
]
honest_scores = [
    balanced_accuracy_lightgbm,
    balanced_accuracy_xgboost,
    balanced_accuracy_catboost,
    balanced_accuracy_blended_ensemble,
]

bar_positions = np.arange(len(model_names))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
inflated_bars = ax.bar(
    bar_positions - bar_width / 2, inflated_scores, bar_width,
    label='Full-set OOF (pseudo-label-inflated)', color='#f4a261',
)
honest_bars = ax.bar(
    bar_positions + bar_width / 2, honest_scores, bar_width,
    label='Honest OOF (original rows only)', color='#2a9d8f',
)
ax.set_xticks(bar_positions)
ax.set_xticklabels(model_names)
ax.set_ylabel('Balanced Accuracy')
ax.set_ylim(0.94, 0.98)
ax.set_title('Inflated vs Honest OOF Balanced Accuracy (Notebook 04)')
ax.legend()
for bar_group in (inflated_bars, honest_bars):
    for bar in bar_group:
        bar_height = bar.get_height()
        ax.annotate(
            f'{bar_height:.4f}',
            (bar.get_x() + bar.get_width() / 2, bar_height),
            ha='center', va='bottom', fontsize=8,
        )
plt.tight_layout()
plt.savefig(outputs_directory / 'inflated_vs_honest_oof.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved to {outputs_directory / "inflated_vs_honest_oof.png"}')


In [ ]:
print('\n' + '=' * 60)
print('TRAINING STACKING META-MODEL')
print('=' * 60)

stacking_meta_model_results = train_stacking_meta_model(
    out_of_fold_predictions_pseudo_labeled_lightgbm,
    out_of_fold_predictions_pseudo_labeled_xgboost,
    out_of_fold_predictions_pseudo_labeled_catboost,
    augmented_training_labels,
    n_splits=5,
)

trained_meta_model = stacking_meta_model_results['meta_model']
mean_cross_validation_score = stacking_meta_model_results['mean_cv_score']
std_cross_validation_score = stacking_meta_model_results['std_cv_score']

print(
    f'\nMeta-Model CV Mean BA: '
    f'{mean_cross_validation_score:.4f} ± {std_cross_validation_score:.4f}'
)

os.makedirs(str(pseudo_labeled_models_directory), exist_ok=True)

save_models(
    pseudo_labeled_lightgbm_models,
    pseudo_labeled_xgboost_models,
    pseudo_labeled_catboost_models,
    output_dir=str(pseudo_labeled_models_directory),
)

meta_model_output_filepath = (
    pseudo_labeled_models_directory / 'meta_model.pkl'
)
save_meta_model(trained_meta_model, str(meta_model_output_filepath))

print(
    f'Saved pseudo-labeled base models to: {pseudo_labeled_models_directory}'
)
print(f'Meta-model saved to: {meta_model_output_filepath}')

In [ ]:
print('\n' + '=' * 60)
print('INFERENCE PIPELINE: HONEST BLEND SEARCH & SUBMISSION')
print('=' * 60)
print("Note: submission is generated purely from this pipeline's own")
print('trained models. No external/reference submission file is loaded')
print('or blended in — every score below is a real, reproducible number')
print('from models trained and validated in this notebook.')

# Wrap raw LightGBM boosters for inference
pseudo_labeled_lightgbm_models = [
    wrap_lgb_model(model)
    if hasattr(model, 'predict') and not hasattr(model, 'predict_proba')
    else model
    for model in pseudo_labeled_lightgbm_models
]
print('Wrapped raw LightGBM boosters for inference')

# Test Time Augmentation (TTA) Predictions
test_probabilities_tta = predict_with_tta(
    pseudo_labeled_lightgbm_models,
    pseudo_labeled_xgboost_models,
    pseudo_labeled_catboost_models,
    test_features,
    noise_level=0.01,
    num_repeats=5,
    w_lgb=0.60,
    w_xgb=0.35,
    w_cat=0.05,
    random_state=42,
)
print(f'TTA predictions shape: {test_probabilities_tta.shape}')

# Base model predictions without TTA
test_predictions_lightgbm = np.zeros((len(test_features), 3))
test_predictions_xgboost = np.zeros((len(test_features), 3))
test_predictions_catboost = np.zeros((len(test_features), 3))

for lightgbm_model in pseudo_labeled_lightgbm_models:
    test_predictions_lightgbm += lightgbm_model.predict_proba(test_features)
test_predictions_lightgbm /= len(pseudo_labeled_lightgbm_models)

for xgboost_model in pseudo_labeled_xgboost_models:
    test_predictions_xgboost += xgboost_model.predict_proba(test_features)
test_predictions_xgboost /= len(pseudo_labeled_xgboost_models)

for catboost_model in pseudo_labeled_catboost_models:
    test_predictions_catboost += catboost_model.predict_proba(test_features)
test_predictions_catboost /= len(pseudo_labeled_catboost_models)

print('Base model predictions aggregated')

print(
    '\nSearching for the best weighted blend, scored only on HONEST OOF predictions'
)
print('(original labeled rows only — excludes the pseudo-labeled rows to avoid')
print('circular/self-confirming evaluation)...')

search_targets = encoded_original_training_targets
search_oof_lightgbm = original_out_of_fold_predictions_lightgbm
search_oof_xgboost = original_out_of_fold_predictions_xgboost
search_oof_catboost = original_out_of_fold_predictions_catboost

MINIMUM_MODEL_WEIGHT = 0.05
TOP_K_CANDIDATES = 15 

candidate_weight_combinations = []
for weight_lightgbm in np.arange(0.0, 1.0 + 1e-9, 0.05):
    for weight_xgboost in np.arange(
        0.0, 1.0 - weight_lightgbm + 1e-9, 0.05
    ):
        weight_catboost = 1.0 - weight_lightgbm - weight_xgboost
        if weight_catboost < 0:
            continue
        weight_lightgbm_rounded = round(float(weight_lightgbm), 2)
        weight_xgboost_rounded = round(float(weight_xgboost), 2)
        weight_catboost_rounded = round(float(weight_catboost), 2)

        if (
            weight_lightgbm_rounded < MINIMUM_MODEL_WEIGHT
            or weight_xgboost_rounded < MINIMUM_MODEL_WEIGHT
            or weight_catboost_rounded < MINIMUM_MODEL_WEIGHT
        ):
            continue

        candidate_weight_combinations.append(
            (
                weight_lightgbm_rounded,
                weight_xgboost_rounded,
                weight_catboost_rounded,
            )
        )

seed_weight_combinations = [
    (0.60, 0.35, 0.05),
    (0.55, 0.35, 0.10),
    (0.50, 0.40, 0.10),
    (0.50, 0.35, 0.15),
    (0.45, 0.35, 0.20),
    (0.40, 0.35, 0.25),
    (0.35, 0.35, 0.30),
    (0.30, 0.35, 0.35),
    (0.20, 0.30, 0.50),
]

for weight_tuple in seed_weight_combinations:
    if (
        all(value >= MINIMUM_MODEL_WEIGHT for value in weight_tuple)
        and weight_tuple not in candidate_weight_combinations
    ):
        candidate_weight_combinations.append(weight_tuple)

scored_weight_combinations = []
for (
    weight_lightgbm,
    weight_xgboost,
    weight_catboost,
) in candidate_weight_combinations:
    blended_train_predictions = (
        search_oof_lightgbm * weight_lightgbm
        + search_oof_xgboost * weight_xgboost
        + search_oof_catboost * weight_catboost
    )
    predicted_class_labels = blended_train_predictions.argmax(axis=1)
    balanced_acc_score = balanced_accuracy_score(
        search_targets, predicted_class_labels
    )
    scored_weight_combinations.append(
        (
            balanced_acc_score,
            weight_lightgbm,
            weight_xgboost,
            weight_catboost,
        )
    )

scored_weight_combinations.sort(key=lambda item: item[0], reverse=True)
top_k_weight_combinations = scored_weight_combinations[:TOP_K_CANDIDATES]

optimal_weight_lightgbm = float(
    np.mean([item[1] for item in top_k_weight_combinations])
)
optimal_weight_xgboost = float(
    np.mean([item[2] for item in top_k_weight_combinations])
)
optimal_weight_catboost = float(
    np.mean([item[3] for item in top_k_weight_combinations])
)


weight_sum = (
    optimal_weight_lightgbm
    + optimal_weight_xgboost
    + optimal_weight_catboost
)
optimal_weight_lightgbm /= weight_sum
optimal_weight_xgboost /= weight_sum
optimal_weight_catboost /= weight_sum


evaluated_blended_train = (
    search_oof_lightgbm * optimal_weight_lightgbm
    + search_oof_xgboost * optimal_weight_xgboost
    + search_oof_catboost * optimal_weight_catboost
)
best_blend_honest_score = balanced_accuracy_score(
    search_targets, evaluated_blended_train.argmax(axis=1)
)

print(
    f'Top-{TOP_K_CANDIDATES} combos HONEST OOF BA range: '
    f'{top_k_weight_combinations[-1][0]:.4f} - {top_k_weight_combinations[0][0]:.4f}'
)

optimal_weight_tuple = (
    round(optimal_weight_lightgbm, 3),
    round(optimal_weight_xgboost, 3),
    round(optimal_weight_catboost, 3),
)

best_blend_test_probabilities = (
    test_predictions_lightgbm * optimal_weight_lightgbm
    + test_predictions_xgboost * optimal_weight_xgboost
    + test_predictions_catboost * optimal_weight_catboost
)

print(
    f'Best blend weights (top-{TOP_K_CANDIDATES} average, min weight {MINIMUM_MODEL_WEIGHT}): '
    f'LGB={optimal_weight_lightgbm:.3f}, XGB={optimal_weight_xgboost:.3f}, CAT={optimal_weight_catboost:.3f}'
)
print(
    f'HONEST OOF balanced accuracy for chosen blend: {best_blend_honest_score:.4f}'
)


meta_model_oof_probabilities = predict_with_meta_model(
    trained_meta_model,
    search_oof_lightgbm,
    search_oof_xgboost,
    search_oof_catboost,
)
meta_model_honest_score = balanced_accuracy_score(
    search_targets, meta_model_oof_probabilities.argmax(axis=1)
)
print(
    f'HONEST OOF balanced accuracy for stacking meta-model: {meta_model_honest_score:.4f}'
)


submission_configurations = [
    ('submission_blend_tta', test_probabilities_tta),
    (
        'submission_lgb60_xgb35_cat05',
        test_predictions_lightgbm * 0.60
        + test_predictions_xgboost * 0.35
        + test_predictions_catboost * 0.05,
    ),
    ('submission_best_blend', best_blend_test_probabilities),
    (
        'submission_meta_model',
        predict_with_meta_model(
            trained_meta_model,
            test_predictions_lightgbm,
            test_predictions_xgboost,
            test_predictions_catboost,
        ),
    ),
]

for configuration_name, predicted_probabilities in submission_configurations:
    submission_dataframe = make_submission_frame(
        raw_test_data['id'],
        predicted_probabilities,
        output_path=str(
            processed_data_directory / f'{configuration_name}.csv'
        ),
    )
    class_distribution = pd.Series(
        [
            REVERSE_MAP[index]
            for index in predicted_probabilities.argmax(axis=1)
        ]
    ).value_counts()

    print(f'\n{configuration_name}:')
    print(f'  File: {configuration_name}.csv')
    print(f'  Predictions shape: {predicted_probabilities.shape}')
    print('  Class distribution:')
    for class_name in ['GALAXY', 'QSO', 'STAR']:
        count = int(class_distribution.get(class_name, 0))
        percentage = 100 * count / len(submission_dataframe)
        print(f'    {class_name:8s}: {count:5d} ({percentage:5.2f}%)')


if meta_model_honest_score >= best_blend_honest_score:
    final_test_probabilities = predict_with_meta_model(
        trained_meta_model,
        test_predictions_lightgbm,
        test_predictions_xgboost,
        test_predictions_catboost,
    )
    selected_model_description = 'stacking meta-model'
    final_selected_oof_score = meta_model_honest_score
else:
    final_test_probabilities = best_blend_test_probabilities
    selected_model_description = (
        f'weighted blend (LGB={optimal_weight_lightgbm:.3f}, '
        f'XGB={optimal_weight_xgboost:.3f}, CAT={optimal_weight_catboost:.3f})'
    )
    final_selected_oof_score = best_blend_honest_score

final_submission_dataframe = make_submission_frame(
    raw_test_data['id'],
    final_test_probabilities,
    output_path=str(processed_data_directory / 'submission.csv'),
)

print(
    f'\nFinal submission.csv written using: {selected_model_description}'
)
print(
    f'  (selected because it had the higher HONEST OOF balanced accuracy: {final_selected_oof_score:.4f})'
)
print(f'\nAll submissions saved to: {processed_data_directory}')

In [ ]:
candidate_labels = [
    (
        f'Weighted blend\n(LGB={optimal_weight_lightgbm:.2f}/'
        f'XGB={optimal_weight_xgboost:.2f}/CAT={optimal_weight_catboost:.2f})'
    ),
    'Stacking meta-model',
]
candidate_scores = [best_blend_honest_score, meta_model_honest_score]
candidate_colors = ['#2a9d8f', '#e76f51']

fig, ax = plt.subplots(figsize=(6.5, 5))
bars = ax.bar(candidate_labels, candidate_scores, color=candidate_colors, width=0.5)
ax.set_ylabel('Honest OOF Balanced Accuracy')
ax.set_ylim(0.94, 0.98)
ax.set_title('Final Blend Candidates: Honest OOF Comparison')
for bar in bars:
    bar_height = bar.get_height()
    ax.annotate(
        f'{bar_height:.4f}',
        (bar.get_x() + bar.get_width() / 2, bar_height),
        ha='center', va='bottom', fontsize=10,
    )
plt.tight_layout()
plt.savefig(outputs_directory / 'blend_vs_meta_model.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved to {outputs_directory / "blend_vs_meta_model.png"}')
print(
    f'Selected: {selected_model_description} '
    f'(honest OOF {final_selected_oof_score:.4f})'
)


In [ ]:
leaderboard_scores = {
    'Notebook 03\n(base blend, LGB .85/XGB 0/CAT .15)': {'private': 0.96520, 'public': 0.96588},
    'Notebook 04\n(pseudo-labeled blend)': {'private': 0.96494, 'public': 0.96545},
}

score_labels = ['Private score', 'Public score']
notebook_03_scores = [leaderboard_scores['Notebook 03\n(base blend, LGB .85/XGB 0/CAT .15)']['private'],
                       leaderboard_scores['Notebook 03\n(base blend, LGB .85/XGB 0/CAT .15)']['public']]
notebook_04_scores = [leaderboard_scores['Notebook 04\n(pseudo-labeled blend)']['private'],
                       leaderboard_scores['Notebook 04\n(pseudo-labeled blend)']['public']]

bar_positions = np.arange(len(score_labels))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
bars_03 = ax.bar(
    bar_positions - bar_width / 2, notebook_03_scores, bar_width,
    label='Notebook 03 (base blend, LGB .85/XGB 0/CAT .15)', color='#264653',
)
bars_04 = ax.bar(
    bar_positions + bar_width / 2, notebook_04_scores, bar_width,
    label='Notebook 04 (pseudo-labeled blend)', color='#e9c46a',
)
ax.set_xticks(bar_positions)
ax.set_xticklabels(score_labels)
ax.set_ylabel('Kaggle Score')
ax.set_ylim(0.963, 0.967)
ax.set_title('Leaderboard Score: Notebook 03 vs Notebook 04')
ax.legend(fontsize=8)
for bar_group in (bars_03, bars_04):
    for bar in bar_group:
        bar_height = bar.get_height()
        ax.annotate(
            f'{bar_height:.5f}',
            (bar.get_x() + bar.get_width() / 2, bar_height),
            ha='center', va='bottom', fontsize=8,
        )
plt.tight_layout()
plt.savefig(outputs_directory / 'leaderboard_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved to {outputs_directory / "leaderboard_comparison.png"}')
